## Notebook 11 — Potential Audience Segment (PAS) Cards

## Purpose

This notebook converts the previously identified baseline clusters into **Potential Audience Segment (PAS) cards** that are readable by analysts and usable by downstream AI systems.

The goal is to transform cluster-level outputs into a structured **baseline audience intelligence layer** for Market Kinetics.

These PAS cards do **not** represent client-specific customer segments. Instead, they describe broad, society-level audience archetypes derived from the integrated baseline data stack:

- **ACS (American Community Survey)** for structural and demographic characteristics  
- **GSS (General Social Survey)** for psychological and attitudinal signals  
- **NPORS / Pew datasets** for media behavior indicators  

This baseline segmentation provides a **societal reference layer** that Market Kinetics can later combine with proprietary customer and behavioral data to identify more refined segments and, ultimately, generate objective-specific **Action Segments (AS)**.

---

## Notebook outputs

This notebook performs the following steps:

1. Converts cluster-level profiles into one row per **Potential Audience Segment (PAS)**  
2. Preserves dominant structural traits derived from ACS  
3. Incorporates psychological signals inferred from the GSS projection layer  
4. Incorporates media behavior signals derived from NPORS / Pew datasets  
5. Generates analyst-readable summaries describing structural, psychological, and media characteristics  
6. Produces a compact **RAG-ready representation** of each segment for downstream retrieval and reasoning workflows  

---

## Expected deliverables

Primary dataset:

- `mk_pas_baseline_v1.parquet`

Optional export formats:

- `mk_pas_baseline_v1.csv` — tabular export for inspection and sharing  
- `mk_pas_rag_corpus_v1.jsonl` — retrieval corpus designed for embedding and vector search pipelines  

---

## Design principle

Each PAS card is designed to function simultaneously as:

- a **human-readable audience intelligence brief**  
- a **machine-readable retrieval object** suitable for downstream AI workflows  

This structure enables Market Kinetics to support both **analyst interpretation** and **AI-assisted audience selection**, campaign planning, and future **Action Segment (AS)** generation once objective-specific data is introduced.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

In [2]:
# Paths
PROJECT_ROOT = Path.cwd().resolve().parents[0]  # adjust if needed

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "societal_processed"
INTERIM_DIR = DATA_DIR / "societal_interim"

OUTPUT_DIR = PROCESSED_DIR / "pas_cards"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/pas_cards


In [3]:
# loading cluster profiles
cluster_profiles_path = PROCESSED_DIR / "10_mk_cluster_profiles_v1.parquet"

df_cluster_profiles = pd.read_parquet(cluster_profiles_path)

print("Loaded:", cluster_profiles_path)
print("Shape:", df_cluster_profiles.shape)

df_cluster_profiles.head()

Loaded: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/10_mk_cluster_profiles_v1.parquet
Shape: (7, 4)


,cluster,population_share,psych_signals,media_signals
0,0,0.128360,None,"[media_usage: internet_frequency, media_usage:..."
1,1,0.177141,[party_alignment: republican],"[media_usage: internet_frequency, media_usage:..."
2,2,0.070639,"[media_engagement: low, media_engagement: high...","[media_usage: tiktok, media_usage: whatsapp, m..."
3,3,0.109562,"[media_engagement: high, media_engagement: low...","[media_usage: snapchat, media_usage: tiktok, m..."
4,4,0.128296,"[media_engagement: low, media_engagement: high...","[media_usage: internet_frequency, media_usage:..."


In [4]:
# loading structural archetypes
MODELS_DIR = PROJECT_ROOT /"data" / "societal_models"
archetypes_path = MODELS_DIR / "06_mk_us_structural_archetypes_v1.csv"

df_archetypes = pd.read_csv(archetypes_path)
df_archetypes.head(7)

,cluster_id,archetype_name,population_pct_total,population_pct_adult,dominant_emp_tier,dominant_hhincome_tier,dominant_tenure,dominant_edu_tier,dominant_mar_tier,dominant_age_bin,dominant_race_eth,dominant_sex_label
0,0,Established Working Families,0.128,0.169,Employed,100-199k,Owner,HS_or_less,Married,35-44,White_NH,Male
1,1,Affluent Mid-Career Homeowners,0.177,0.233,Employed,100-199k,Owner,Some_college,Married,55-64,White_NH,Female
2,2,Kids in Working Families,0.071,0.038,Missing,100-199k,Owner,HS_or_less,Never_Married,6-12,Hispanic,Male
3,3,Early-Career Homeowners,0.110,0.103,Employed,100-199k,Owner,HS_or_less,Never_Married,18-24,White_NH,Female
4,4,Retired Renters,0.128,0.166,Retired,50-99k,Renter,HS_or_less,Married,65+,White_NH,Male
5,5,Dependent Kids in Shared Homes,0.207,0.067,Missing,50-99k,No_Rent,HS_or_less,Never_Married,6-12,White_NH,Male
6,6,Struggling Working Renters,0.179,0.225,Employed,20-49k,No_Rent,HS_or_less,Never_Married,25-34,White_NH,Female


In [5]:
# Define columns for PAS cards
pas_columns = [
    "segment_id",
    "cluster_id",
    "archetype_name",
    "baseline_layer_version",
    "population_share",
    "population_rank", # leave empty in this phase. to be generated at the end of the process.

    "dominant_age_bin",
    "dominant_sex_label",
    "dominant_race_eth",
    "dominant_edu_tier",
    "dominant_emp_tier",
    "dominant_income_tier",
    "dominant_mar_tier",
    "dominant_tenure",
    "structural_profile",

    "psych_signals",
    "psych_summary",
    "media_signals",
    "media_summary",

    "motivational_drivers", # those emerging from baseline layer. to be updated after behavior data is ingested
    "key_barriers", # those emerging from baseline layer. to be updated after behavior data is ingested
    "trust_cues", # those emerging from baseline layer. to be updated after behavior data is ingested
    "channel_implications", # those emerging from baseline layer. to be updated after behavior data is ingested
    "messaging_implications", # those emerging from baseline layer. to be updated after behavior data is ingested
    "offer_implications", # those emerging from baseline layer. to be updated after behavior data is ingested
    "susceptibility_notes", # to be generated after the Supporting OBJ has been created

    "narrative_summary",
    "tags",
    "source_layers",
    "rag_text",
]

In [6]:
# creating a PAS dataframe
# empty PAS dataframe with final schema
df_pas = pd.DataFrame(columns=pas_columns)

df_pas.head()

,segment_id,cluster_id,archetype_name,baseline_layer_version,population_share,population_rank,dominant_age_bin,dominant_sex_label,dominant_race_eth,dominant_edu_tier,...,key_barriers,trust_cues,channel_implications,messaging_implications,offer_implications,susceptibility_notes,narrative_summary,tags,source_layers,rag_text


In [7]:
df_cluster_profiles.head()

,cluster,population_share,psych_signals,media_signals
0,0,0.128360,None,"[media_usage: internet_frequency, media_usage:..."
1,1,0.177141,[party_alignment: republican],"[media_usage: internet_frequency, media_usage:..."
2,2,0.070639,"[media_engagement: low, media_engagement: high...","[media_usage: tiktok, media_usage: whatsapp, m..."
3,3,0.109562,"[media_engagement: high, media_engagement: low...","[media_usage: snapchat, media_usage: tiktok, m..."
4,4,0.128296,"[media_engagement: low, media_engagement: high...","[media_usage: internet_frequency, media_usage:..."


In [8]:
print(df_cluster_profiles.loc[4, "psych_signals"])

['media_engagement: low' 'media_engagement: high' 'ideology: conservative']


In [9]:
# ---------------------------------------------------
# Build PAS base table
# ---------------------------------------------------

df_pas = pd.DataFrame({
    "segment_id": df_cluster_profiles["cluster"].apply(lambda x: f"PAS_{int(x):02d}"),
    "cluster_id": df_cluster_profiles["cluster"],
    "archetype_name": None,  # to be filled later
    "baseline_layer_version": "v1",
    "population_share": df_cluster_profiles["population_share"],
    "population_rank": None,  # calculated later

    "dominant_age_bin": None,
    "dominant_sex_label": None,
    "dominant_race_eth": None,
    "dominant_edu_tier": None,
    "dominant_emp_tier": None,
    "dominant_income_tier": None,
    "dominant_mar_tier": None,
    "dominant_tenure": None,

    "structural_profile": None,

    "psych_signals": df_cluster_profiles["psych_signals"],
    "psych_summary": None,

    "media_signals": df_cluster_profiles["media_signals"],
    "media_summary": None,

    "motivational_drivers": None,
    "key_barriers": None,
    "trust_cues": None,
    "channel_implications": None,
    "messaging_implications": None,
    "offer_implications": None,
    "susceptibility_notes": None,

    "narrative_summary": None,
    "tags": None,
    "source_layers": None,
    "rag_text": None,
})

df_pas.head()

,segment_id,cluster_id,archetype_name,baseline_layer_version,population_share,population_rank,dominant_age_bin,dominant_sex_label,dominant_race_eth,dominant_edu_tier,...,key_barriers,trust_cues,channel_implications,messaging_implications,offer_implications,susceptibility_notes,narrative_summary,tags,source_layers,rag_text
0,PAS_00,0,None,v1,0.128360,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,PAS_01,1,None,v1,0.177141,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,PAS_02,2,None,v1,0.070639,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,PAS_03,3,None,v1,0.109562,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,PAS_04,4,None,v1,0.128296,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [10]:
# Merge archetypes data into PAS dataframe

df_pas = df_pas.merge(
    df_archetypes,
    on="cluster_id",
    how="left"
)

df_pas.head()

,segment_id,cluster_id,archetype_name_x,baseline_layer_version,population_share,population_rank,dominant_age_bin_x,dominant_sex_label_x,dominant_race_eth_x,dominant_edu_tier_x,...,population_pct_total,population_pct_adult,dominant_emp_tier_y,dominant_hhincome_tier,dominant_tenure_y,dominant_edu_tier_y,dominant_mar_tier_y,dominant_age_bin_y,dominant_race_eth_y,dominant_sex_label_y
0,PAS_00,0,None,v1,0.128360,None,None,None,None,None,...,0.128,0.169,Employed,100-199k,Owner,HS_or_less,Married,35-44,White_NH,Male
1,PAS_01,1,None,v1,0.177141,None,None,None,None,None,...,0.177,0.233,Employed,100-199k,Owner,Some_college,Married,55-64,White_NH,Female
2,PAS_02,2,None,v1,0.070639,None,None,None,None,None,...,0.071,0.038,Missing,100-199k,Owner,HS_or_less,Never_Married,6-12,Hispanic,Male
3,PAS_03,3,None,v1,0.109562,None,None,None,None,None,...,0.110,0.103,Employed,100-199k,Owner,HS_or_less,Never_Married,18-24,White_NH,Female
4,PAS_04,4,None,v1,0.128296,None,None,None,None,None,...,0.128,0.166,Retired,50-99k,Renter,HS_or_less,Married,65+,White_NH,Male


In [11]:
df_pas.columns

Index(['segment_id', 'cluster_id', 'archetype_name_x',
       'baseline_layer_version', 'population_share', 'population_rank',
       'dominant_age_bin_x', 'dominant_sex_label_x', 'dominant_race_eth_x',
       'dominant_edu_tier_x', 'dominant_emp_tier_x', 'dominant_income_tier',
       'dominant_mar_tier_x', 'dominant_tenure_x', 'structural_profile',
       'psych_signals', 'psych_summary', 'media_signals', 'media_summary',
       'motivational_drivers', 'key_barriers', 'trust_cues',
       'channel_implications', 'messaging_implications', 'offer_implications',
       'susceptibility_notes', 'narrative_summary', 'tags', 'source_layers',
       'rag_text', 'archetype_name_y', 'population_pct_total',
       'population_pct_adult', 'dominant_emp_tier_y', 'dominant_hhincome_tier',
       'dominant_tenure_y', 'dominant_edu_tier_y', 'dominant_mar_tier_y',
       'dominant_age_bin_y', 'dominant_race_eth_y', 'dominant_sex_label_y'],
      dtype='str')

In [12]:
# renaming columns
df_pas = df_pas.rename(columns={
    "population_pct_adult": "population_adult_share",
    "dominant_hhincome_tier": "dominant_household_income_tier"
})

In [13]:
# drop redundant columns from archetypes
df_pas = df_pas.drop(columns=[
    "population_pct_total"
])

df_pas = df_pas.drop(columns=["dominant_income_tier"])

In [14]:
# ---------------------------------------------------
# Clean merged PAS columns after archetype merge
# ---------------------------------------------------

df_pas = df_pas.drop(columns=[
    "archetype_name_x",
    "dominant_age_bin_x",
    "dominant_sex_label_x",
    "dominant_race_eth_x",
    "dominant_edu_tier_x",
    "dominant_emp_tier_x",
    "dominant_mar_tier_x",
    "dominant_tenure_x",
])

df_pas = df_pas.rename(columns={
    "archetype_name_y": "archetype_name",
    "dominant_age_bin_y": "dominant_age_bin",
    "dominant_sex_label_y": "dominant_sex_label",
    "dominant_race_eth_y": "dominant_race_eth",
    "dominant_edu_tier_y": "dominant_edu_tier",
    "dominant_emp_tier_y": "dominant_emp_tier",
    "dominant_mar_tier_y": "dominant_mar_tier",
    "dominant_tenure_y": "dominant_tenure",
})

In [15]:
df_pas.columns

Index(['segment_id', 'cluster_id', 'baseline_layer_version',
       'population_share', 'population_rank', 'structural_profile',
       'psych_signals', 'psych_summary', 'media_signals', 'media_summary',
       'motivational_drivers', 'key_barriers', 'trust_cues',
       'channel_implications', 'messaging_implications', 'offer_implications',
       'susceptibility_notes', 'narrative_summary', 'tags', 'source_layers',
       'rag_text', 'archetype_name', 'population_adult_share',
       'dominant_emp_tier', 'dominant_household_income_tier',
       'dominant_tenure', 'dominant_edu_tier', 'dominant_mar_tier',
       'dominant_age_bin', 'dominant_race_eth', 'dominant_sex_label'],
      dtype='str')

In [16]:
df_pas.head()

,segment_id,cluster_id,baseline_layer_version,population_share,population_rank,structural_profile,psych_signals,psych_summary,media_signals,media_summary,...,archetype_name,population_adult_share,dominant_emp_tier,dominant_household_income_tier,dominant_tenure,dominant_edu_tier,dominant_mar_tier,dominant_age_bin,dominant_race_eth,dominant_sex_label
0,PAS_00,0,v1,0.128360,None,None,None,None,"[media_usage: internet_frequency, media_usage:...",None,...,Established Working Families,0.169,Employed,100-199k,Owner,HS_or_less,Married,35-44,White_NH,Male
1,PAS_01,1,v1,0.177141,None,None,[party_alignment: republican],None,"[media_usage: internet_frequency, media_usage:...",None,...,Affluent Mid-Career Homeowners,0.233,Employed,100-199k,Owner,Some_college,Married,55-64,White_NH,Female
2,PAS_02,2,v1,0.070639,None,None,"[media_engagement: low, media_engagement: high...",None,"[media_usage: tiktok, media_usage: whatsapp, m...",None,...,Kids in Working Families,0.038,Missing,100-199k,Owner,HS_or_less,Never_Married,6-12,Hispanic,Male
3,PAS_03,3,v1,0.109562,None,None,"[media_engagement: high, media_engagement: low...",None,"[media_usage: snapchat, media_usage: tiktok, m...",None,...,Early-Career Homeowners,0.103,Employed,100-199k,Owner,HS_or_less,Never_Married,18-24,White_NH,Female
4,PAS_04,4,v1,0.128296,None,None,"[media_engagement: low, media_engagement: high...",None,"[media_usage: internet_frequency, media_usage:...",None,...,Retired Renters,0.166,Retired,50-99k,Renter,HS_or_less,Married,65+,White_NH,Male


In [17]:
df_pas.columns

Index(['segment_id', 'cluster_id', 'baseline_layer_version',
       'population_share', 'population_rank', 'structural_profile',
       'psych_signals', 'psych_summary', 'media_signals', 'media_summary',
       'motivational_drivers', 'key_barriers', 'trust_cues',
       'channel_implications', 'messaging_implications', 'offer_implications',
       'susceptibility_notes', 'narrative_summary', 'tags', 'source_layers',
       'rag_text', 'archetype_name', 'population_adult_share',
       'dominant_emp_tier', 'dominant_household_income_tier',
       'dominant_tenure', 'dominant_edu_tier', 'dominant_mar_tier',
       'dominant_age_bin', 'dominant_race_eth', 'dominant_sex_label'],
      dtype='str')

In [18]:
pas_columns = [
    "segment_id",
    "cluster_id",
    "archetype_name",
    "baseline_layer_version",
    "population_share",
    "population_adult_share",
    "population_rank",

    "dominant_age_bin",
    "dominant_sex_label",
    "dominant_race_eth",
    "dominant_edu_tier",
    "dominant_emp_tier",
    "dominant_household_income_tier",
    "dominant_mar_tier",
    "dominant_tenure",
    "structural_profile",

    "psych_signals",
    "psych_summary",
    "media_signals",
    "media_summary",

    "motivational_drivers",
    "key_barriers",
    "trust_cues",
    "channel_implications",
    "messaging_implications",
    "offer_implications",
    "susceptibility_notes",

    "narrative_summary",
    "tags",
    "source_layers",
    "rag_text",
]

In [19]:
df_pas = df_pas[pas_columns]

df_pas.head()

,segment_id,cluster_id,archetype_name,baseline_layer_version,population_share,population_adult_share,population_rank,dominant_age_bin,dominant_sex_label,dominant_race_eth,...,key_barriers,trust_cues,channel_implications,messaging_implications,offer_implications,susceptibility_notes,narrative_summary,tags,source_layers,rag_text
0,PAS_00,0,Established Working Families,v1,0.128360,0.169,None,35-44,Male,White_NH,...,None,None,None,None,None,None,None,None,None,None
1,PAS_01,1,Affluent Mid-Career Homeowners,v1,0.177141,0.233,None,55-64,Female,White_NH,...,None,None,None,None,None,None,None,None,None,None
2,PAS_02,2,Kids in Working Families,v1,0.070639,0.038,None,6-12,Male,Hispanic,...,None,None,None,None,None,None,None,None,None,None
3,PAS_03,3,Early-Career Homeowners,v1,0.109562,0.103,None,18-24,Female,White_NH,...,None,None,None,None,None,None,None,None,None,None
4,PAS_04,4,Retired Renters,v1,0.128296,0.166,None,65+,Male,White_NH,...,None,None,None,None,None,None,None,None,None,None


In [20]:
# Clean race/ethnicity labels by removing "_NH"
# Example: "White_NH" -> "White"

df_pas["dominant_race_eth"] = (
    df_pas["dominant_race_eth"]
    .astype(str)
    .str.replace("_NH", "", regex=False)
)

In [21]:
df_pas[["segment_id", "dominant_race_eth"]]

,segment_id,dominant_race_eth
0,PAS_00,White
1,PAS_01,White
2,PAS_02,Hispanic
3,PAS_03,White
4,PAS_04,White
5,PAS_05,White
6,PAS_06,White


### PAS Card Initialization — Structural Backbone Integration

In this phase, the **initial PAS card structure** was created and populated with the baseline information available at the cluster level.

First, the cluster-level intelligence dataset (`10_mk_cluster_profiles_v1.parquet`) was used to generate the **PAS skeleton**, establishing one row per segment and preserving key baseline attributes such as:

- `segment_id`
- `cluster_id`
- `population_share`
- `psych_signals`
- `media_signals`

Next, the **structural archetype layer** (`df_archetypes`) was merged into the PAS table. This step added the dominant demographic and socioeconomic traits associated with each cluster, including:

- dominant age group
- sex
- race/ethnicity
- education level
- employment status
- household income tier
- marital status
- housing tenure

Redundant columns created during the merge were cleaned, and column names were standardized to match the PAS schema. The final dataframe was then reordered according to the defined PAS card structure.

At this stage, the PAS table represents a **structural and behavioral backbone** for each baseline segment of the U.S. population. Interpretation and narrative fields remain empty and will be generated in subsequent steps.

### Next Step

The next phase will generate the **`structural_profile`** field, which converts the dominant structural attributes of each segment into a concise, human-readable description. This will serve as the first narrative component of each PAS card and will later support the construction of `narrative_summary` and `rag_text`.

In [22]:
# building structural profile narrative based on dominant demographic attributes
def build_structural_profile(row):

    parts = []

    if pd.notna(row["dominant_age_bin"]):
        parts.append(f"primarily aged {row['dominant_age_bin']}")

    if pd.notna(row["dominant_emp_tier"]):
        parts.append(f"{row['dominant_emp_tier'].lower()}")

    if pd.notna(row["dominant_mar_tier"]):
        parts.append(f"{row['dominant_mar_tier'].replace('_',' ').lower()}")

    if pd.notna(row["dominant_edu_tier"]):
        parts.append(f"with {row['dominant_edu_tier'].replace('_',' ').lower()} education")

    if pd.notna(row["dominant_household_income_tier"]):
        parts.append(f"household income typically in the {row['dominant_household_income_tier']} range")

    if pd.notna(row["dominant_tenure"]):
        parts.append(f"mostly {row['dominant_tenure'].lower()}s")

    if pd.notna(row["dominant_race_eth"]):
        parts.append(f"predominantly {row['dominant_race_eth'].replace('_',' ')}")

    return "Segment composed of individuals " + ", ".join(parts) + "."

In [23]:
# apply
df_pas["structural_profile"] = df_pas.apply(build_structural_profile, axis=1)

df_pas[[
    "segment_id",
    "archetype_name",
    "structural_profile"
]].head()

,segment_id,archetype_name,structural_profile
0,PAS_00,Established Working Families,Segment composed of individuals primarily aged...
1,PAS_01,Affluent Mid-Career Homeowners,Segment composed of individuals primarily aged...
2,PAS_02,Kids in Working Families,Segment composed of individuals primarily aged...
3,PAS_03,Early-Career Homeowners,Segment composed of individuals primarily aged...
4,PAS_04,Retired Renters,Segment composed of individuals primarily aged...


In [24]:
df_pas.columns

Index(['segment_id', 'cluster_id', 'archetype_name', 'baseline_layer_version',
       'population_share', 'population_adult_share', 'population_rank',
       'dominant_age_bin', 'dominant_sex_label', 'dominant_race_eth',
       'dominant_edu_tier', 'dominant_emp_tier',
       'dominant_household_income_tier', 'dominant_mar_tier',
       'dominant_tenure', 'structural_profile', 'psych_signals',
       'psych_summary', 'media_signals', 'media_summary',
       'motivational_drivers', 'key_barriers', 'trust_cues',
       'channel_implications', 'messaging_implications', 'offer_implications',
       'susceptibility_notes', 'narrative_summary', 'tags', 'source_layers',
       'rag_text'],
      dtype='str')

### CLAUDE INTEGRATION
Using claude to generate pyschological, media and narrative summaries for the clusters.

In [25]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [26]:
# PROMPT FOR PSYCH SIGNALS SUMMARY
PSYCH_PROMPT = """
You are an audience intelligence analyst working on a baseline segmentation of the United States population.

Your task is to interpret the psychological signals associated with one audience segment and produce a short analytical summary.

The signals represent statistically meaningful psychological or attitudinal indicators associated with the segment relative to the broader population.

Your goal is to translate these signals into a clear, human-readable description of the segment’s likely attitudinal tendencies.

Guidelines:

• Write 2–3 concise sentences.
• Use neutral analytical language (not marketing language).
• Use cautious phrasing such as "appears", "suggests", "may indicate", or "is associated with".
• Preserve all important signals, especially ideological or political cues if present.
• If signals appear mixed or contradictory, acknowledge the ambiguity rather than collapsing them.
• Do not invent traits not supported by the signals.
• Do not provide recommendations or messaging.

Segment information:

Segment name:
{archetype_name}

Psychological signals detected in the cluster:
{psych_signals}

Task:

Write a concise analytical description of the psychological tendencies of this audience segment based strictly on the signals provided.

Output only the summary text.
Do not include headings, bullet points, or explanations.
"""

In [27]:
# function to generate psych summary using the prompt and the model
def normalize_signals(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, list):
        cleaned = [str(v).strip() for v in x if str(v).strip()]
        return ", ".join(cleaned) if cleaned else None
    x = str(x).strip()
    return x if x else None

In [28]:
# iterate over PAS dataframe and generate psych summary for each row
for i, row in df_pas.iterrows():

    psych_text = normalize_signals(row["psych_signals"])

    if psych_text is None:
        df_pas.loc[i, "psych_summary"] = None
        continue

    prompt = PSYCH_PROMPT.format(
        archetype_name=row["archetype_name"],
        psych_signals=psych_text
    )

    response = _client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )

    df_pas.loc[i, "psych_summary"] = response.content[0].text.strip()

In [29]:
# Display the relevant columns to review the generated summaries
df_pas[["segment_id", "archetype_name", "psych_signals", "psych_summary"]]

,segment_id,archetype_name,psych_signals,psych_summary
0,PAS_00,Established Working Families,None,None
1,PAS_01,Affluent Mid-Career Homeowners,[party_alignment: republican],This segment appears to lean toward Republican...
2,PAS_02,Kids in Working Families,"[media_engagement: low, media_engagement: high...",This segment appears to lean Republican in its...
3,PAS_03,Early-Career Homeowners,"[media_engagement: high, media_engagement: low...",This segment appears to report moderate life s...
4,PAS_04,Retired Renters,"[media_engagement: low, media_engagement: high...",This segment is associated with a conservative...
5,PAS_05,Dependent Kids in Shared Homes,"[media_engagement: low, media_engagement: high...",This segment appears to lean Republican in its...
6,PAS_06,Struggling Working Renters,"[party_alignment: republican, party_alignment:...",The Struggling Working Renters segment appears...


In [30]:
# Example output for one segment
row = df_pas[df_pas["segment_id"] == "PAS_04"].iloc[0]

print("Psych signals:")
print(row["psych_signals"])
print()
print("Psych summary:")
print(row["psych_summary"])

Psych signals:
['media_engagement: low' 'media_engagement: high' 'ideology: conservative']

Psych summary:
This segment is associated with a conservative ideological orientation, suggesting alignment with traditional or right-leaning values and perspectives. The media engagement signals appear contradictory, with both low and high engagement indicated, which may reflect internal variation within the segment or context-dependent consumption patterns rather than a uniform tendency. This ambiguity makes it difficult to characterize the group's relationship with media consumption with confidence, though the conservative ideological signal remains a consistent marker.


In [31]:
### Media summary prompt
MEDIA_PROMPT = """
You are an audience intelligence analyst working on a baseline segmentation of the United States population.

Your task is to interpret the media signals associated with one audience segment and produce a short analytical summary.

The signals represent statistically meaningful media usage or information-consumption indicators associated with the segment relative to the broader population.

Your goal is to translate these signals into a clear, human-readable description of how this segment appears to consume information and interact with media environments.

Guidelines:

• Write 2–3 concise sentences.
• Use neutral analytical language (not marketing language).
• Use cautious phrasing such as "appears", "suggests", "may indicate", or "is associated with".
• Preserve all important channel cues if present.
• If signals appear mixed or contradictory, acknowledge the ambiguity rather than collapsing them.
• Do not invent behaviors not supported by the signals.
• Do not provide recommendations or messaging.

Segment information:

Segment name:
{archetype_name}

Media signals detected in the cluster:
{media_signals}

Task:

Write a concise analytical description of the media behavior of this audience segment based strictly on the signals provided.

Output only the summary text.
Do not include headings, bullet points, or explanations.
"""

In [32]:
# generate media summary
for i, row in df_pas.iterrows():

    media_text = normalize_signals(row["media_signals"])

    if media_text is None:
        df_pas.loc[i, "media_summary"] = None
        continue

    prompt = MEDIA_PROMPT.format(
        archetype_name=row["archetype_name"],
        media_signals=media_text
    )

    response = _client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )

    df_pas.loc[i, "media_summary"] = response.content[0].text.strip()

In [33]:
# check media summaries
df_pas[["segment_id", "archetype_name", "media_signals", "media_summary"]]

,segment_id,archetype_name,media_signals,media_summary
0,PAS_00,Established Working Families,"[media_usage: internet_frequency, media_usage:...",This segment appears to be consistently active...
1,PAS_01,Affluent Mid-Career Homeowners,"[media_usage: internet_frequency, media_usage:...",This segment appears to be consistently connec...
2,PAS_02,Kids in Working Families,"[media_usage: tiktok, media_usage: whatsapp, m...",This segment appears to be concentrated in soc...
3,PAS_03,Early-Career Homeowners,"[media_usage: snapchat, media_usage: tiktok, m...",This segment appears to engage heavily with sh...
4,PAS_04,Retired Renters,"[media_usage: internet_frequency, media_usage:...",This segment appears to engage with digital me...
5,PAS_05,Dependent Kids in Shared Homes,"[media_usage: tiktok, media_usage: snapchat, m...",This segment's media behavior appears strongly...
6,PAS_06,Struggling Working Renters,"[media_usage: internet_frequency, media_usage:...",This segment appears to be heavily oriented to...


In [34]:
# Example output for one segment
row = df_pas[df_pas["segment_id"] == "PAS_04"].iloc[0]

print("Media signals:")
print(row["media_signals"])
print()
print("Media summary:")
print(row["media_summary"])

Media signals:
['media_usage: internet_frequency' 'media_usage: instagram'
 'media_usage: tiktok']

Media summary:
This segment appears to engage with digital media on a regular basis, suggesting consistent internet access and online activity despite being a retired population. The presence of both Instagram and TikTok signals may indicate an affinity for visually-driven, social media platforms typically associated with younger demographics, which could reflect broader adoption of these platforms across age groups. The combination of general internet frequency with these specific social channels suggests a media profile that, while rooted in digital habits, may warrant cautious interpretation given the potential ambiguity between primary and incidental platform use.


In [35]:
# Narrative summary prompt
NARRATIVE_PROMPT = """
You are an audience intelligence analyst working on a baseline segmentation of the United States population.

Your task is to write a concise audience narrative for one baseline segment.

The goal is to produce a human-readable analytical description that integrates structural, psychological, and media-behavior evidence into a coherent profile of the segment.

This is not marketing copy and not a persuasion message. It is an analytical segment description suitable for an audience intelligence brief.

Guidelines:

• Write 4–6 sentences in one paragraph.
• Use clear, natural, professional prose.
• Use neutral analytical language.
• Use cautious phrasing such as "appears", "suggests", "may indicate", or "is associated with".
• Ground the narrative strictly in the evidence provided.
• Preserve important ideological, attitudinal, and media-use cues if present.
• If signals are mixed or somewhat contradictory, acknowledge the ambiguity rather than forcing false coherence.
• Do not invent motivations, barriers, or recommendations.
• Do not give messaging advice.
• Do not mention "the dataset", "the cluster", or "the model".

Segment information:

Segment name:
{archetype_name}

Population share:
{population_share}

Adult population share:
{population_adult_share}

Structural profile:
{structural_profile}

Psychological signals:
{psych_signals}

Psychological summary:
{psych_summary}

Media signals:
{media_signals}

Media summary:
{media_summary}

Task:

Write one concise analytical paragraph describing this audience segment as a baseline social group within U.S. society.

Output only the paragraph text.
Do not include headings, bullet points, or explanations.
"""

In [36]:
# Generate narrative summary for each segment
for i, row in df_pas.iterrows():

    psych_text = normalize_signals(row["psych_signals"])
    media_text = normalize_signals(row["media_signals"])

    prompt = NARRATIVE_PROMPT.format(
        segment_id=row["segment_id"],
        archetype_name=row["archetype_name"],
        population_share=f"{row['population_share']:.1%}" if pd.notna(row["population_share"]) else "N/A",
        population_adult_share=f"{row['population_adult_share']:.1%}" if pd.notna(row["population_adult_share"]) else "N/A",
        structural_profile=row["structural_profile"] if pd.notna(row["structural_profile"]) else "Not available.",
        psych_signals=psych_text if psych_text is not None else "No strong psychological signals detected.",
        psych_summary=row["psych_summary"] if pd.notna(row["psych_summary"]) else "No clear psychological summary available.",
        media_signals=media_text if media_text is not None else "No strong media signals detected.",
        media_summary=row["media_summary"] if pd.notna(row["media_summary"]) else "No clear media summary available."
    )

    response = _client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )

    df_pas.loc[i, "narrative_summary"] = response.content[0].text.strip()

In [37]:
# Display the narrative summaries
df_pas[[
    "segment_id",
    "archetype_name",
    "narrative_summary"
]]

,segment_id,archetype_name,narrative_summary
0,PAS_00,Established Working Families,Established Working Families represents a segm...
1,PAS_01,Affluent Mid-Career Homeowners,Affluent Mid-Career Homeowners represent appro...
2,PAS_02,Kids in Working Families,"This segment, labeled Kids in Working Families..."
3,PAS_03,Early-Career Homeowners,Early-Career Homeowners represent approximatel...
4,PAS_04,Retired Renters,Retired Renters represent approximately 12.8% ...
5,PAS_05,Dependent Kids in Shared Homes,This segment represents a structurally distinc...
6,PAS_06,Struggling Working Renters,Struggling Working Renters represent approxima...


In [38]:
# Example output for one segment
row = df_pas[df_pas["segment_id"] == "PAS_04"].iloc[0]

print("Narrative summary:")
print(row["narrative_summary"])

Narrative summary:
Retired Renters represent approximately 12.8% of the U.S. population and 16.6% of the adult population, composed predominantly of individuals aged 65 and older who are retired, married, and White, with educational attainment at the high school level or below and household incomes generally falling in the $50,000–$99,000 range — a profile that suggests modest but stable financial circumstances in later life, with renter status potentially reflecting either regional housing market conditions or constrained asset accumulation. The segment is associated with a conservative ideological orientation, suggesting alignment with traditional or right-leaning values, though this signal alone cannot fully characterize the internal diversity likely present within a group of this demographic breadth. Media engagement signals are notably contradictory, with both low and high engagement indicated, which may reflect meaningful variation within the segment rather than a uniform consump

### Generate Narrative Summaries

This step synthesizes the structural, psychological, and media evidence associated with each PAS into a single analyst-readable narrative paragraph.

The narrative summary integrates multiple layers of evidence at once: structural profile, psychological signals and summaries, media signals and summaries, and segment population share. This produces a more coherent baseline description of each segment as a social group within U.S. society.

The resulting `narrative_summary` serves as the core descriptive text of the PAS card and will later support downstream retrieval, audience selection, and strategy-generation workflows.

In [39]:
pd.set_option("display.max_columns", None)
df_pas.head()

,segment_id,cluster_id,archetype_name,baseline_layer_version,population_share,population_adult_share,population_rank,dominant_age_bin,dominant_sex_label,dominant_race_eth,dominant_edu_tier,dominant_emp_tier,dominant_household_income_tier,dominant_mar_tier,dominant_tenure,structural_profile,psych_signals,psych_summary,media_signals,media_summary,motivational_drivers,key_barriers,trust_cues,channel_implications,messaging_implications,offer_implications,susceptibility_notes,narrative_summary,tags,source_layers,rag_text
0,PAS_00,0,Established Working Families,v1,0.128360,0.169,None,35-44,Male,White,HS_or_less,Employed,100-199k,Married,Owner,Segment composed of individuals primarily aged...,None,None,"[media_usage: internet_frequency, media_usage:...",This segment appears to be consistently active...,None,None,None,None,None,None,None,Established Working Families represents a segm...,None,None,None
1,PAS_01,1,Affluent Mid-Career Homeowners,v1,0.177141,0.233,None,55-64,Female,White,Some_college,Employed,100-199k,Married,Owner,Segment composed of individuals primarily aged...,[party_alignment: republican],This segment appears to lean toward Republican...,"[media_usage: internet_frequency, media_usage:...",This segment appears to be consistently connec...,None,None,None,None,None,None,None,Affluent Mid-Career Homeowners represent appro...,None,None,None
2,PAS_02,2,Kids in Working Families,v1,0.070639,0.038,None,6-12,Male,Hispanic,HS_or_less,Missing,100-199k,Never_Married,Owner,Segment composed of individuals primarily aged...,"[media_engagement: low, media_engagement: high...",This segment appears to lean Republican in its...,"[media_usage: tiktok, media_usage: whatsapp, m...",This segment appears to be concentrated in soc...,None,None,None,None,None,None,None,"This segment, labeled Kids in Working Families...",None,None,None
3,PAS_03,3,Early-Career Homeowners,v1,0.109562,0.103,None,18-24,Female,White,HS_or_less,Employed,100-199k,Never_Married,Owner,Segment composed of individuals primarily aged...,"[media_engagement: high, media_engagement: low...",This segment appears to report moderate life s...,"[media_usage: snapchat, media_usage: tiktok, m...",This segment appears to engage heavily with sh...,None,None,None,None,None,None,None,Early-Career Homeowners represent approximatel...,None,None,None
4,PAS_04,4,Retired Renters,v1,0.128296,0.166,None,65+,Male,White,HS_or_less,Retired,50-99k,Married,Renter,Segment composed of individuals primarily aged...,"[media_engagement: low, media_engagement: high...",This segment is associated with a conservative...,"[media_usage: internet_frequency, media_usage:...",This segment appears to engage with digital me...,None,None,None,None,None,None,None,Retired Renters represent approximately 12.8% ...,None,None,None


### Populate Source Layers

This step records the baseline data layers used to construct each PAS card.

At this stage, all segments are derived from the same integrated baseline stack:

- **ACS** for structural and demographic attributes
- **GSS** for psychological and attitudinal signals
- **Pew / NPORS** for media behavior signals

The `source_layers` field preserves provenance and makes the PAS cards easier to audit, filter, and reuse in later retrieval and strategy-generation workflows.

In [40]:
# adding source layers info
df_pas["source_layers"] = [
    ["ACS_structural", "GSS_psych", "Pew_NPORS_media"]
    for _ in range(len(df_pas))
]

In [41]:
# quick check
df_pas[["segment_id", "archetype_name", "source_layers"]].head()

,segment_id,archetype_name,source_layers
0,PAS_00,Established Working Families,"[ACS_structural, GSS_psych, Pew_NPORS_media]"
1,PAS_01,Affluent Mid-Career Homeowners,"[ACS_structural, GSS_psych, Pew_NPORS_media]"
2,PAS_02,Kids in Working Families,"[ACS_structural, GSS_psych, Pew_NPORS_media]"
3,PAS_03,Early-Career Homeowners,"[ACS_structural, GSS_psych, Pew_NPORS_media]"
4,PAS_04,Retired Renters,"[ACS_structural, GSS_psych, Pew_NPORS_media]"


### Generate PAS Tags

This step creates a compact set of machine-readable tags for each PAS card.

The tags are derived from dominant structural attributes and the available psychological and media signals. Their purpose is not to replace the narrative summary, but to provide a lightweight indexing layer that supports filtering, retrieval, and later segment selection workflows.

At this stage, tags remain descriptive and baseline-oriented. They do not encode objective-specific judgments or campaign recommendations.

In [42]:
# Tags creating function
def normalize_signals_to_list(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    x = str(x).strip()
    return [x] if x else []


def make_pas_tags(row):
    tags = []

    # structural tags
    structural_fields = [
        "dominant_age_bin",
        "dominant_sex_label",
        "dominant_race_eth",
        "dominant_edu_tier",
        "dominant_emp_tier",
        "dominant_household_income_tier",
        "dominant_mar_tier",
        "dominant_tenure",
    ]

    for col in structural_fields:
        val = row.get(col)
        if pd.notna(val):
            tags.append(str(val).lower())

    # psych tags
    psych_tags = normalize_signals_to_list(row.get("psych_signals"))
    tags.extend([t.lower().replace(": ", "_").replace(":", "_").replace(" ", "_") for t in psych_tags])

    # media tags
    media_tags = normalize_signals_to_list(row.get("media_signals"))
    tags.extend([t.lower().replace(": ", "_").replace(":", "_").replace(" ", "_") for t in media_tags])

    # remove duplicates while preserving order
    seen = set()
    clean_tags = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            clean_tags.append(tag)

    return clean_tags

In [43]:
# apply the function to create tags for each segment
df_pas["tags"] = df_pas.apply(make_pas_tags, axis=1)

In [44]:
# quick check
df_pas[["segment_id", "archetype_name", "tags"]].head()

,segment_id,archetype_name,tags
0,PAS_00,Established Working Families,"[35-44, male, white, hs_or_less, employed, 100..."
1,PAS_01,Affluent Mid-Career Homeowners,"[55-64, female, white, some_college, employed,..."
2,PAS_02,Kids in Working Families,"[6-12, male, hispanic, hs_or_less, missing, 10..."
3,PAS_03,Early-Career Homeowners,"[18-24, female, white, hs_or_less, employed, 1..."
4,PAS_04,Retired Renters,"[65+, male, white, hs_or_less, retired, 50-99k..."


### Generate RAG Text

This step creates a retrieval-ready text representation of each PAS card.

The `rag_text` field condenses the most important descriptive elements of the segment into a single structured paragraph, combining identity, population relevance, structural profile, psychological interpretation, media interpretation, and the overall narrative summary.

This field is designed for downstream retrieval workflows, enabling later components of Market Kinetics to search, compare, and reason over PAS cards as reusable audience intelligence objects.

In [45]:
RAG_INTERPRETATION_PROMPT = """
You are an audience intelligence analyst.

Your task is to write a short retrieval-oriented interpretation of one audience segment for use in a retrieval-augmented generation system.

The goal is not to restate all details already provided. Instead, produce a compact analytical synthesis that captures the segment’s overall character in 1–2 sentences.

Guidelines:
- Write exactly 1–2 sentences.
- Use neutral analytical language.
- Use cautious phrasing such as "appears", "suggests", or "may indicate".
- Focus on the segment’s overall character and any notable tensions or heterogeneity.
- Do not repeat population size, structural details, or platform names unless necessary.
- Do not invent unsupported motivations, recommendations, or messaging advice.
- Output only the interpretation text.

Segment name:
{archetype_name}

Structural profile:
{structural_profile}

Psychological profile:
{psych_summary}

Media behavior:
{media_summary}
"""

In [46]:
# helper function to generate RAG interpretation text for each segment
def generate_rag_interpretation(row):
    prompt = RAG_INTERPRETATION_PROMPT.format(
        archetype_name=row["archetype_name"],
        structural_profile=row["structural_profile"] if pd.notna(row["structural_profile"]) else "Not available.",
        psych_summary=row["psych_summary"] if pd.notna(row["psych_summary"]) else "Not available.",
        media_summary=row["media_summary"] if pd.notna(row["media_summary"]) else "Not available."
    )

    response = _client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=120,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.content[0].text.strip()

In [47]:
# function to generate RAG text for each segment

def build_rag_text(row):
    parts = []

    # identity
    parts.append(f"Segment ID: {row['segment_id']}")
    parts.append(f"Segment name: {row['archetype_name']}")

    # population relevance
    pop_bits = []
    if pd.notna(row.get("population_share")):
        pop_bits.append(f"{row['population_share']:.1%} of the U.S. population")
    if pd.notna(row.get("population_adult_share")):
        pop_bits.append(f"{row['population_adult_share']:.1%} of the adult population")
    if pop_bits:
        parts.append("Population relevance: " + "; ".join(pop_bits))

    # structural
    if pd.notna(row.get("structural_profile")) and row["structural_profile"]:
        parts.append(f"Structural profile: {str(row['structural_profile']).strip()}")

    # psych
    if pd.notna(row.get("psych_summary")) and row["psych_summary"]:
        parts.append(f"Psychological profile: {str(row['psych_summary']).strip()}")

    # media
    if pd.notna(row.get("media_summary")) and row["media_summary"]:
        parts.append(f"Media behavior: {str(row['media_summary']).strip()}")

    # short LLM interpretation
    interpretation = generate_rag_interpretation(row)
    parts.append(f"Interpretation: {interpretation}")

    return "\n".join(parts)

In [48]:
# apply the function to create RAG text for each segment
df_pas["rag_text"] = df_pas.apply(build_rag_text, axis=1)

In [49]:
# quick check
row = df_pas.iloc[4]

print(row["rag_text"])

Segment ID: PAS_04
Segment name: Retired Renters
Population relevance: 12.8% of the U.S. population; 16.6% of the adult population
Structural profile: Segment composed of individuals primarily aged 65+, retired, married, with hs or less education, household income typically in the 50-99k range, mostly renters, predominantly White.
Psychological profile: This segment is associated with a conservative ideological orientation, suggesting alignment with traditional or right-leaning values and perspectives. The media engagement signals appear contradictory, with both low and high engagement indicated, which may reflect internal variation within the segment or context-dependent consumption patterns rather than a uniform tendency. This ambiguity makes it difficult to characterize the group's relationship with media consumption with confidence, though the conservative ideological signal remains a consistent marker.
Media behavior: This segment appears to engage with digital media on a regular 

### PAS Dataset Quality Check

This step performs a basic integrity check on the PAS dataset before export.

The check verifies that all segments contain the required analytical fields:
- structural profile
- psychological summary
- media summary
- narrative summary
- rag text representation

Any missing values are flagged so they can be reviewed before finalizing the baseline PAS layer.

In [50]:
# columns that must be populated
required_cols = [
    "structural_profile",
    "psych_summary",
    "media_summary",
    "narrative_summary",
    "rag_text"
]

# check for missing values
qa_missing = df_pas[required_cols].isnull().sum()

print("Missing values by column:")
print(qa_missing)

Missing values by column:
structural_profile    0
psych_summary         1
media_summary         0
narrative_summary     0
rag_text              0
dtype: int64


In [51]:
rows_with_issues = df_pas[df_pas[required_cols].isnull().any(axis=1)]

print("Rows with missing required fields:")
print(rows_with_issues[["segment_id", "archetype_name"]])

Rows with missing required fields:
  segment_id                archetype_name
0     PAS_00  Established Working Families


**Note:** Segment `PAS_00 – Established Working Families` has no `psych_summary` because no psychological traits in this cluster deviated significantly from the national baseline in the GSS projection layer.  
This indicates that the segment’s attitudinal profile closely mirrors the overall population rather than exhibiting distinctive psychological signals.

In [53]:
# EXPORTING THE PAS CARDS BASELINE TABLE
pas_path = OUTPUT_DIR / "mk_pas_baseline_v1.parquet"
df_pas.to_parquet(pas_path, index=False)

print("Saved:", pas_path)

Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/pas_cards/mk_pas_baseline_v1.parquet


In [54]:
# exporting to csv
csv_path = OUTPUT_DIR / "mk_pas_baseline_v1.csv"
df_pas.to_csv(csv_path, index=False)

print("Saved:", csv_path)

Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/pas_cards/mk_pas_baseline_v1.csv


In [55]:
# RAG corpus
import json

rag_path = OUTPUT_DIR / "mk_pas_rag_corpus_v1.jsonl"

with open(rag_path, "w") as f:
    for _, row in df_pas.iterrows():
        record = {
            "segment_id": row["segment_id"],
            "archetype_name": row["archetype_name"],
            "tags": row["tags"],
            "rag_text": row["rag_text"]
        }
        f.write(json.dumps(record) + "\n")

print("Saved:", rag_path)

Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/pas_cards/mk_pas_rag_corpus_v1.jsonl


### Notebook Summary

This notebook converted baseline societal clusters into structured **Potential Audience Segment (PAS) cards**, integrating structural (ACS), psychological (GSS), and media behavior (NPORS/Pew) signals.

The resulting dataset represents the **Market Kinetics baseline audience intelligence layer**, providing both analyst-readable segment profiles and RAG-ready representations for downstream retrieval, targeting, and future Target Audience generation.